# USFM Embedding Visualization (t-SNE / UMAP)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import LabelEncoder

try:
    import umap
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '--quiet'])
    import umap

# ── Path config ──────────────────────────────────────────────
PATH_LAST3      = Path('/gpfs/gibbs/project/papademetris/jg3279/USFM/week8/newdatafinetuning_organ_last3layers')
PATH_EXTRABLOCK = Path('/gpfs/gibbs/project/papademetris/jg3279/USFM/week8/extra_block_only')
PATH_HFE_ORIG   = Path('/gpfs/gibbs/project/papademetris/jg3279/USFM/week8/feature_level_hfe')
PATH_HFE_B      = Path('/gpfs/gibbs/project/papademetris/jg3279/USFM/week9/ablation_B_hfe_alpha1')
PATH_MLP        = Path('/gpfs/gibbs/project/papademetris/jg3279/USFM/week9/ablation_A_extrablock_mlp')
PATH_FROZEN     = Path('/gpfs/gibbs/project/papademetris/jg3279/USFM/week9/ablation_E_extrablock_frozenall')

OUTPUT_VIZ = Path('/gpfs/gibbs/project/papademetris/jg3279/USFM/week9/visualizations')
OUTPUT_VIZ.mkdir(parents=True, exist_ok=True)

ORGAN_COLORS = {
    'breast':  '#E74C3C',
    'carotid': '#3498DB',
    'kidney':  '#2ECC71',
    'liver':   '#F39C12',
    'thyroid': '#9B59B6',
}
ORGAN_ORDER = ['breast', 'carotid', 'kidney', 'liver', 'thyroid']


In [ ]:
def load_test(emb_path, organs_path, splits_path=None):
    """Load test split embeddings only."""
    emb = np.load(emb_path)
    organs = np.load(organs_path)
    if splits_path is not None:
        splits = np.load(splits_path)
        mask = splits == 'test'
        return emb[mask], organs[mask]
    return emb, organs

DATA = {
    'Last3Layers': load_test(
        PATH_LAST3 / 'embeddings.npy',
        PATH_LAST3 / 'organs.npy',
        PATH_LAST3 / 'splits.npy',
    ),
    'ExtraBlock': load_test(
        PATH_EXTRABLOCK / 'embeddings_test.npy',
        PATH_EXTRABLOCK / 'organs_test.npy',
    ),
    'HFE_original': load_test(
        PATH_HFE_ORIG / 'embeddings_test.npy',
        PATH_HFE_ORIG / 'organs_test.npy',
    ),
    'HFE_B': load_test(
        PATH_HFE_B / 'embeddings_test.npy',
        PATH_HFE_B / 'organs_test.npy',
    ),
    'EquivMLP': load_test(
        PATH_MLP / 'embeddings_test.npy',
        PATH_MLP / 'organs_test.npy',
    ),
    'FrozenAll': load_test(
        PATH_FROZEN / 'embeddings_test.npy',
        PATH_FROZEN / 'organs_test.npy',
    ),
}

for name, (emb, org) in DATA.items():
    print(f'{name:20s}: {emb.shape}  organs={org.shape}')


In [ ]:
def compute_tsne(emb):
    # Compatibility across sklearn versions (n_jobs may be unavailable)
    try:
        return TSNE(
            n_components=2,
            perplexity=30,
            n_iter=1000,
            random_state=42,
            n_jobs=-1,
        ).fit_transform(emb)
    except TypeError:
        return TSNE(
            n_components=2,
            perplexity=30,
            n_iter=1000,
            random_state=42,
        ).fit_transform(emb)

RESULTS = {}
for name, (emb, organs) in DATA.items():
    print(f'Computing {name}...', end=' ', flush=True)
    tsne_2d = compute_tsne(emb)
    umap_2d = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        random_state=42,
    ).fit_transform(emb)
    RESULTS[name] = {
        'emb': emb,
        'organs': organs,
        'tsne': tsne_2d,
        'umap': umap_2d,
    }
    print('✓')
print('Done')


In [ ]:
def scatter(ax, coords, organs, title):
    for organ in ORGAN_ORDER:
        mask = organs == organ
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=ORGAN_COLORS[organ],
            s=18,
            alpha=0.7,
            edgecolors='none',
        )
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Test Set Embedding Space: Last3Layers vs ExtraBlock', fontsize=14, fontweight='bold')

scatter(axes[0, 0], RESULTS['Last3Layers']['tsne'], RESULTS['Last3Layers']['organs'], 'Last3Layers — t-SNE  (80.65%)')
scatter(axes[0, 1], RESULTS['ExtraBlock']['tsne'],  RESULTS['ExtraBlock']['organs'],  'ExtraBlock Only — t-SNE  (95.81%)')
scatter(axes[1, 0], RESULTS['Last3Layers']['umap'], RESULTS['Last3Layers']['organs'], 'Last3Layers — UMAP  (80.65%)')
scatter(axes[1, 1], RESULTS['ExtraBlock']['umap'],  RESULTS['ExtraBlock']['organs'],  'ExtraBlock Only — UMAP  (95.81%)')

handles = [mpatches.Patch(color=ORGAN_COLORS[o], label=o.capitalize()) for o in ORGAN_ORDER]
fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=11, frameon=False, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig(OUTPUT_VIZ / 'fig1_extrablock_vs_last3.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved fig1')


In [ ]:
panels = [
    ('Last3Layers', '(a) Last3Layers\n80.65%'),
    ('ExtraBlock',  '(b) ExtraBlock Only\n95.81%'),
    ('FrozenAll',   '(c) ExtraBlock, Backbone Frozen\n97.10%'),
    ('HFE_B',       '(d) ExtraBlock + HFE\n98.39%'),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Embedding Space Progression — t-SNE (Test Set)', fontsize=14, fontweight='bold')

for ax, (name, title) in zip(axes, panels):
    scatter(ax, RESULTS[name]['tsne'], RESULTS[name]['organs'], title)

handles = [mpatches.Patch(color=ORGAN_COLORS[o], label=o.capitalize()) for o in ORGAN_ORDER]
fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=11, frameon=False, bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
plt.savefig(OUTPUT_VIZ / 'fig2_progression.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved fig2')


In [ ]:
le = LabelEncoder()
print(f"{'Experiment':25s} {'Raw Emb':>10} {'t-SNE':>10} {'UMAP':>10}")
print('-' * 60)

for name in ['Last3Layers', 'ExtraBlock', 'FrozenAll', 'HFE_original', 'HFE_B']:
    r = RESULTS[name]
    y = le.fit_transform(r['organs'])
    s_raw = silhouette_score(r['emb'], y)
    s_tsne = silhouette_score(r['tsne'], y)
    s_umap = silhouette_score(r['umap'], y)
    print(f'{name:25s} {s_raw:10.4f} {s_tsne:10.4f} {s_umap:10.4f}')
